# Neural-network robustness certification (auto_LiRPA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/fmaiv/blob/main/day04/examples/nn/robustness.ipynb)

**FMAIV Day 4 — the frontier hands-on.** Same *verification* question as the rest of the week
("can the bad thing happen?"), now for a neural network:

> Within an L-infinity ball of radius `eps` around an input `x0`, can the classifier's prediction change?

We use **auto_LiRPA** — the CROWN bound-propagation engine underneath
[alpha,beta-CROWN](https://github.com/Verified-Intelligence/alpha-beta-CROWN), the VNN-COMP winner.
It computes a **certified** lower bound on the margin `z[true] - z[other]` over the *entire* ball.
If that bound is `> 0`, **no** input in the ball is misclassified — a proof, not a sample.

Everything here is tiny and **CPU-only** (a 2-D, 2-class MLP), so it runs in ~1s on the free Colab
tier. The same code applies unchanged to a trained MNIST/CIFAR network — see our
[AAAI'26 VNN-COMP tutorial](https://vnn-comp.github.io/#aaai2026) for full-scale notebooks.


## 0. Install

On Colab `torch` is already present, so we only add `auto_LiRPA`. Its maintained
release lives on GitHub (PyPI is stuck at an ancient 0.2/0.3), pinned here for reproducibility.


In [ ]:
!pip -q install git+https://github.com/Verified-Intelligence/auto_LiRPA.git@ca767f1d8c0a6b125a292ba165adb2319bbaf615

## 1. A tiny dataset and model
Two well-separated Gaussian blobs (2 classes) and a small ReLU MLP. Deterministic via a fixed seed.


In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)

def make_data(n=600):
    half = n // 2
    blob0 = torch.randn(half, 2) * 0.7 + torch.tensor([1.0, 1.0])   # class 0
    blob1 = torch.randn(half, 2) * 0.7 + torch.tensor([-1.0, -1.0]) # class 1 (some overlap)
    X = torch.cat([blob0, blob1], 0)
    y = torch.cat([torch.zeros(half), torch.ones(half)]).long()
    return X, y

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2,16), nn.ReLU(),
                                 nn.Linear(16,16), nn.ReLU(),
                                 nn.Linear(16,2))
    def forward(self, x):
        return self.net(x)

X, y = make_data()
model = MLP()
opt = torch.optim.Adam(model.parameters(), lr=0.05)
lossf = nn.CrossEntropyLoss()
for _ in range(300):
    opt.zero_grad(); lossf(model(X), y).backward(); opt.step()
model.eval()
acc = (model(X).argmax(1) == y).float().mean().item()
print(f'train accuracy: {acc:.3f}')

## 2. Certify the margin under an L-inf perturbation
`compute_bounds(..., C=C, method='CROWN')` returns a certified lower bound on the linear
specification `C @ logits`. We set `C` to pick out the margin `z[true] - z[other]`.
A lower bound `> 0` means **certified robust** at that `eps`.


In [ ]:
from auto_LiRPA import BoundedModule, BoundedTensor
from auto_LiRPA.perturbations import PerturbationLpNorm

# A correctly-classified point with a moderate (not razor-thin) clean margin, so it
# certifies for a band of small eps and then breaks as the ball grows.
with torch.no_grad():
    logits = model(X)
    correct = logits.argmax(1) == y
    margins = torch.where(correct,
        logits.gather(1, y.view(-1,1)).squeeze(1) - logits.gather(1, (1-y).view(-1,1)).squeeze(1),
        torch.full_like(logits[:,0], float('inf')))
    idx = int((margins - 3.5).abs().argmin())
x0 = X[idx:idx+1]
true_cls = y[idx].item()
other = 1 - true_cls
lirpa_model = BoundedModule(model, torch.empty_like(x0))

C = torch.zeros(1, 1, 2)
C[0, 0, true_cls] = 1.0
C[0, 0, other] = -1.0

print(f'input = {[round(v,3) for v in x0.tolist()[0]]}, true class = {true_cls}')
print(f"{'eps':>6} {'cert. margin':>13}  verdict")
for eps in [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]:
    ptb = PerturbationLpNorm(norm=float('inf'), eps=eps)
    bx = BoundedTensor(x0, ptb)
    lb, ub = lirpa_model.compute_bounds(x=(bx,), C=C, method='CROWN')
    m = lb.item()
    verdict = 'CERTIFIED ROBUST' if m > 0 else 'not certified by CROWN'
    print(f'{eps:6.2f} {m:13.4f}  {verdict}')

## 3. Visualize: the boxes, the decision, and CROWN's incompleteness
Two panels.

**Left — input space.** The network's decision regions, the point `x0`, and ℓ∞ boxes at several `eps`, each colored by what we can actually conclude:
- **green** — CROWN *certifies* robustness (the whole box is one class);
- **amber** — CROWN *cannot* certify, yet a dense search finds **no** counterexample, so the box is in fact still robust — this gap is CROWN's **incompleteness**;
- **red** — a genuine **counterexample** exists; the ✕ (joined to `x0`) is a concrete input in the box that the network assigns to the *other* class, across the decision boundary.

Points are colored by the network's **prediction** (so they match the regions): robustness is about what the network *predicts* across the box, not about the training labels — a training point's true label can differ from the network's prediction without being a counterexample.

**Right — output space (why the boxes are colored that way).** The class is `argmax`, i.e. the **margin** `z[true] − z[other] > 0`. Plotted against `eps`: the **true** worst-case margin over the box, and two *sound lower bounds* on it — **CROWN** (a linear relaxation) and the looser **IBP** (axis-aligned intervals). A box certifies only while a bound stays **above 0**. The bounds sit *below* the true margin (soundness); the looser the set representation, the sooner it dips below 0 and gives up — IBP first, CROWN later — while the true margin only turns negative once a real counterexample appears. The gap between a bound and the truth is over-approximation; tightening it (better relaxations, zonotopes / star sets, branch-and-bound) is what verifiers compete on.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from auto_LiRPA import BoundedTensor
from auto_LiRPA.perturbations import PerturbationLpNorm

px, py = x0[0].tolist()
GREEN, AMBER, RED = '#1a7f37', '#cc8800', '#c0392b'

# Sound lower bound on the margin z[true]-z[other] over the eps-box, by method.
def margin_lb(eps, method):
    ptb = PerturbationLpNorm(norm=float('inf'), eps=eps)
    lb, _ = lirpa_model.compute_bounds(x=(BoundedTensor(x0, ptb),), C=C, method=method)
    return lb.item()

# TRUE worst-case margin over the box (dense 2-D sweep) and the worst point — a
# falsifier when that margin is < 0. Cheap here because the network is tiny and 2-D.
def worst_case(eps, k=120):
    xs = torch.linspace(px - eps, px + eps, k); ys = torch.linspace(py - eps, py + eps, k)
    gx, gy = torch.meshgrid(xs, ys, indexing='xy')
    pts = torch.stack([gx.reshape(-1), gy.reshape(-1)], 1)
    with torch.no_grad():
        margin = model(pts)[:, true_cls] - model(pts)[:, other]
    j = int(margin.argmin())
    return margin[j].item(), pts[j].tolist()

# Decision regions (network argmax) over a background grid.
gxr = np.linspace(-3.5, 3.5, 320); GX, GY = np.meshgrid(gxr, gxr)
bg = torch.tensor(np.stack([GX.ravel(), GY.ravel()], 1), dtype=torch.float32)
with torch.no_grad():
    region = model(bg).argmax(1).numpy().reshape(GX.shape)
    data_pred = model(X).argmax(1).numpy()      # color points by PREDICTION, so they match the regions

eps_show = [0.2, 0.45, 0.9]
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 5.4))

# ---- Panel A: input space — boxes colored by what we can actually conclude ----
axL.contourf(GX, GY, region, alpha=0.16, levels=1, cmap='coolwarm')
axL.contour(GX, GY, region, levels=[0.5], colors='#222', linewidths=1.4, zorder=4)  # decision boundary
axL.scatter(X[:, 0], X[:, 1], c=data_pred, s=6, cmap='coolwarm', alpha=0.26, zorder=1)
cex = None
for e in eps_show:
    lb = margin_lb(e, 'CROWN'); wmin, wpt = worst_case(e)
    color = GREEN if lb > 0 else (RED if wmin < 0 else AMBER)   # certified / counterexample / incompleteness
    if wmin < 0 and lb <= 0:
        cex = wpt
    axL.add_patch(Rectangle((px - e, py - e), 2*e, 2*e, fill=False, edgecolor=color, lw=2.6, zorder=6))
    axL.text(px - e, py + e + 0.04, f'eps={e}', color=color, ha='left', va='bottom',
             fontsize=9, fontweight='bold', zorder=7)
axL.scatter([px], [py], marker='*', s=250, edgecolor='k', facecolor='gold', linewidths=1.1, zorder=8)
if cex is not None:                              # a real counterexample exists for the largest box
    axL.plot([px, cex[0]], [py, cex[1]], color=RED, lw=1.0, ls='--', zorder=5)
    axL.scatter([cex[0]], [cex[1]], marker='X', s=200, edgecolor='k', facecolor=RED, linewidths=1.2, zorder=9)
    axL.annotate('counterexample\n(predicted other class)', (cex[0], cex[1]),
                 textcoords='offset points', xytext=(8, 4), fontsize=8, color=RED, fontweight='bold')
pad = max(eps_show) + 0.6
axL.set_xlim(px - pad, px + pad); axL.set_ylim(py - pad, py + pad)
axL.set_title('Input space: decision regions + L-inf boxes around x0')
axL.set_xlabel('x1'); axL.set_ylabel('x2')
axL.legend(handles=[
    Line2D([0], [0], marker='*', color='w', markerfacecolor='gold', markeredgecolor='k', markersize=14, label='x0'),
    Line2D([0], [0], marker='X', color='w', markerfacecolor=RED, markeredgecolor='k', markersize=11, label='counterexample'),
    Line2D([0], [0], color='#222', lw=1.4, label='decision boundary'),
    Line2D([0], [0], color=GREEN, lw=2.6, label='certified robust (CROWN)'),
    Line2D([0], [0], color=AMBER, lw=2.6, label='not certified, no counterexample'),
    Line2D([0], [0], color=RED, lw=2.6, label='counterexample exists'),
], loc='lower right', fontsize=7)

# ---- Panel B: output space — the margin the decision actually depends on ----
eg = np.linspace(0, 1.0, 41)
crown = [margin_lb(e, 'CROWN') for e in eg]
ibp   = [margin_lb(e, 'IBP')   for e in eg]
truem = [worst_case(e, 90)[0]  for e in eg]
axR.axhline(0, color='k', lw=1, ls='--')
axR.plot(eg, truem, color='#333',    lw=2.3, label='true min margin (worst case in box)')
axR.plot(eg, crown, color='#1f6feb', lw=2.0, label='CROWN bound (linear relaxation)')
axR.plot(eg, ibp,   color='#8250df', lw=2.0, ls=':', label='IBP bound (intervals)')
eC = max([e for e, c in zip(eg, crown) if c > 0], default=0.0)   # CROWN certifies while bound > 0
eT = next((e for e, t in zip(eg, truem) if t < 0), eg[-1])       # first real counterexample
axR.axvspan(0,  eC,      color=GREEN, alpha=0.12)
axR.axvspan(eC, eT,      color=AMBER, alpha=0.12)
axR.axvspan(eT, eg[-1],  color=RED,   alpha=0.12)
axR.set_ylim(-6, 4.2); axR.set_xlim(0, 1.0)
axR.set_title('Output space: the decision is argmax  <=>  margin > 0')
axR.set_xlabel('eps (L-inf radius)'); axR.set_ylabel('margin   z[true] - z[other]')
axR.legend(fontsize=7, loc='lower left')

plt.tight_layout(); plt.show()
print(f"CROWN certifies up to eps={eC:.2f}; first real counterexample at eps={eT:.2f} "
      f"(the band in between is CROWN's incompleteness gap).")

## Takeaways
- **Certified (green) = a proof** of robustness over the whole box — sound, like the CBMC/Lean verdicts earlier in the week.
- **"Not certified" (amber) does NOT mean "vulnerable."** CROWN is *incomplete*: its bound can fall below 0 while the box is still robust (no counterexample). The amber band is exactly that gap.
- **Counterexample (red) = genuinely non-robust:** a concrete input in the box is assigned to the other class (the box crosses the decision boundary). Falsifiers — projected gradient descent (PGD), or the dense search used here — find it.
- **Verification happens in the output space.** The decision is `argmax`, i.e. the sign of the margin; a verifier soundly *bounds* that margin over the entire input box. Tighter set representations shrink the over-approximation and recover completeness: intervals (IBP) ⊂ linear bounds / zonotopes (CROWN) ⊂ star sets / branch-and-bound (complete, e.g. α,β-CROWN). Same soundness-vs-completeness story as the rest of the course — now for neural networks.

**Next:** the script form is `robustness.py` (and `robustness_starter.py` blanks the `compute_bounds` call). For full-scale MNIST/CIFAR verification and competition benchmarks, see the [AAAI'26 VNN-COMP tutorial](https://vnn-comp.github.io/#aaai2026).